# 55 — Form-defined families (Task 6, exploratory)

Tests for the convergence of a dominant design on families declared by **form** (wing, tilt, booms, propulsor band, enclosure), not by architecture.
Exploratory, after Task 1, on the **w15** weighting and nothing else. Nothing here changes F or any number of the dissertation: F stays as decided on 2026-10-14.
The brief named this notebook 53; 53 and 54 already exist, so it is 55.

All the logic is in `viz/thesis_form_families.py` (pure functions over `ctx`, each cached on the context); this notebook only calls it, shows each table and writes the outputs.

| step | function | output |
|---|---|---|
| 6.1 form variables | `form_variables` | `q8_form_variables.csv` |
| 6.2 declared families | `FAMILIES`, `membership` | (rules fixed before running) |
| 6.3a membership | `membership_tables`, `crosstab` | `q8_families_membership.csv` |
| 6.3b premise | `premise_check`, `q_decomposition` | `q8_families_premise.csv`, `q8_families_q_decomposition.csv` |
| 6.3c condition 1 | `condition1` | `q8_families_condition1.csv` |
| 6.3d condition 2 | `condition2` | `q8_families_condition2.csv` |
| 6.3e condition 3 | `condition3` (quoted from Task 1) | `q8_families_condition3.csv` |
| 6.4 verdict | `verdict_table`, `numbers_md` | `q8_families_verdict.csv`, `NUMBERS_VI_form_families.md` |

Outputs: `thesis_figures_v2/Part_III_dominant_design/III3_form_families/exploratory/`, copies in `LaTeX_v15/Data_for_writing/csv/` (md in `LaTeX_v15/Data_for_writing/`) and in `handoff_2026-10-16/task6/` at the repository root.


## Setup

In [1]:
import sys, pathlib, importlib
PILLAR = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(PILLAR))
import pandas as pd
from IPython.display import display, Markdown
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60); pd.set_option('display.max_colwidth', 120)
from viz import config, thesis
ff = importlib.import_module('viz.thesis_form_families')
cfg = config.load()
ctx = thesis.context(cfg)
OUT = ff.out_dir(cfg)
print(len(ctx.v), 'designs | cohorts', ctx.WIN_NAMES, '| seed', ctx.SEED, '| rarefy', cfg.analysis['rarefy_n'], '| permutations', ff.PERMS, '| writing to', OUT)

627 designs | cohorts ['<= 2011', '2012-15', '2016-19', '2020-23', '2024-26 (partial)'] | seed 42 | rarefy 40 | permutations 200 | writing to /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_analysis/labelling_analysis_thesis/thesis_figures_v2/Part_III_dominant_design/III3_form_families/exploratory


/home/vasco/Vasco Workspace/Tese_Vasco_Lnx/eVTOL-Visual-Evaluation/labeling_evaluation/viz/frames.py:560: UserWarning: rapidfuzz not installed — fuzzy company matching disabled. Only exact lookup table matches will be applied.
  canon = g._normalise_company(sraw)


## 6.1 Form variables

One row per design with an architecture (625). `boom_n` is the raw boom total (not the w15 rank); the band is the w15 propulsor band as a label; the rest come from the w15 frame. A blank means an override hides the reading or the design has no propulsor card / count.

In [2]:
fv = ff.form_variables(ctx)
print(fv.shape, '| blanks:', {c: int(n) for c, n in fv.isna().sum().items() if n})
display(fv.head(10))
for c in ['wing_n', 'wing_tilts', 'band', 'enclosure', 'empType']:
    print(c, fv[c].value_counts(dropna=False).to_dict())

(625, 14) | blanks: {'wing_n': 1, 'wing_tilts': 1, 'band': 30, 'band_rank': 30, 'tilt_share': 31, 'enclosure': 31, 'mounts': 31, 'empType': 26}


,aircraft_id,topType,wing_n,wing_tilts,boom_n,band,band_rank,tilt_share,enclosure,mounts,empType,year,window,in_force
0,US2021331794A1_ua1,TR,2.0,no,0,4,1.0,1.0,Open,wing,VertFin,2018,2016-19,True
1,US2023056709A1_ua1,MR,0.0,no,3,5-6,2.0,1.0,Open,boom,Tailless,2020,2020-23,True
2,US2023056709A1_ua2,MR,0.0,no,2,4,1.0,1.0,Open,boom,Tailless,2020,2020-23,True
3,US2018244367A1_ua1,MR,0.0,no,2,4,1.0,0.0,Open,boom,Tailless,2017,2016-19,True
4,US2018265189A1_ua1,MR,0.0,no,2,4,1.0,0.0,Open,boom,Tailless,2017,2016-19,True
5,EP3770063A1_ua1,MR,0.0,no,2,4,1.0,1.0,Ducted,boom,Tailless,2019,2016-19,True
6,EP3770063A1_ua2,MR,0.0,no,3,5-6,2.0,1.0,Ducted,boom,Tailless,2019,2016-19,True
7,US2019291863A1_ua1,TW,2.0,yes,0,7-8,3.0,1.0,Open,wing,Fins,2016,2016-19,False
8,US2016236774A1_ua1,SLC,2.0,no,1,5-6,2.0,0.0,Open,boom/fuselage,V-Tail,2015,2012-15,True
9,US2016347447A1_ua1,SLC,2.0,no,0,5-6,2.0,0.0,Open,fuselage/wing,VertFin,2015,2012-15,True


wing_n {1.0: 357, 2.0: 172, 0.0: 79, 3.0: 16, nan: 1}
wing_tilts {'no': 551, 'yes': 73, None: 1}
band {'1-3': 136, '4': 125, '5-6': 117, '9+': 117, '7-8': 100, None: 30}
enclosure {'Open': 408, 'Ducted': 186, None: 31}
empType {'Tailless': 176, 'Conventional': 74, 'V-Tail': 69, 'VertFin': 60, 'T-Tail': 56, 'Fins': 51, 'H-Tail': 48, None: 26, 'HorizStab': 21, 'BoxTail': 19, 'Oth': 12, 'Inv_V-Tail': 8, 'Cruciform': 4, 'Y-Tail': 1}


## 6.2 Declared families

Fixed before running; none added or tuned after seeing results. Empennage type is never a membership rule. A design whose rule variable is blank is not a member.

In [3]:
display(ff.declared_families())
mem = ff.membership(ctx)
print('members:', mem[ff.FAMILY_IDS].sum().to_dict())
print('not members for a blank rule variable:', {f: int(mem[f + '_blank'].sum()) for f in ff.FAMILY_IDS})

,id,name,rule,premise
0,P1,"fixed wing, boom-borne distributed propulsion",wing_n >= 1 AND wing_tilts = no AND boom_n >= 1 AND band >= 5-6,"F in form terms: a lifting wing for range, many small propulsors on booms; unit tilt free"
1,P2,"fixed wing, distributed propulsion",wing_n >= 1 AND wing_tilts = no AND band >= 5-6,P1 without the boom rule
2,P3,"fixed wing, open distributed propulsion",P2 AND enclosure = not ducted,ducted propulsors are a distinct form
3,P4,"wingless multicopter, distributed",wing_n = 0 AND band >= 5-6 AND enclosure = not ducted,the Multicopter counterpart of P3
4,P5,"fixed wing with distributed propulsion, any tilt",wing_n >= 1 AND band >= 5-6,"upper bound, includes Tilt Wing and Tilt Frame"


members: {'P1': 174, 'P2': 264, 'P3': 192, 'P4': 14, 'P5': 303}
not members for a blank rule variable: {'P1': 30, 'P2': 30, 'P3': 31, 'P4': 31, 'P5': 30}


## 6.3a Membership

Family x architecture (codes), designs per cohort and the empennage distribution inside each family, on both bases. A family more than 85 % one architecture is that architecture renamed.

In [4]:
mt = ff.membership_tables(ctx)
display(mt['summary'].drop(columns=['architectures (code: designs)']))
print('all designs'); display(ff.crosstab(ctx, 'all designs'))
print('actively protected'); display(ff.crosstab(ctx, 'actively protected'))
long = mt['long']
display(long[long['kind'].eq('cohort') & long['base'].eq('all designs')].pivot(index='family', columns='value', values='designs')[ctx.WIN_NAMES])
display(long[long['kind'].eq('empennage') & long['base'].eq('all designs')].pivot(index='family', columns='value', values='designs').fillna(0).astype(int))

,family,name,base,designs in base,members,share of base,not members for a blank rule variable,top architecture,top architecture designs,top architecture share,renamed architecture (> 85 %)
0,P1,"fixed wing, boom-borne distributed propulsion",all designs,625,174,0.2784,30,SLC,90,0.5172,False
1,P1,"fixed wing, boom-borne distributed propulsion",actively protected,433,135,0.3118,19,SLC,69,0.5111,False
2,P2,"fixed wing, distributed propulsion",all designs,625,264,0.4224,30,SLC,127,0.4811,False
3,P2,"fixed wing, distributed propulsion",actively protected,433,199,0.4596,19,SLC,90,0.4523,False
4,P3,"fixed wing, open distributed propulsion",all designs,625,192,0.3072,31,SLC,101,0.5260,False
5,P3,"fixed wing, open distributed propulsion",actively protected,433,148,0.3418,20,SLC,76,0.5135,False
6,P4,"wingless multicopter, distributed",all designs,625,14,0.0224,31,MR,10,0.7143,False
7,P4,"wingless multicopter, distributed",actively protected,433,12,0.0277,20,MR,8,0.6667,False
8,P5,"fixed wing with distributed propulsion, any tilt",all designs,625,303,0.4848,30,SLC,127,0.4191,False
9,P5,"fixed wing with distributed propulsion, any tilt",actively protected,433,231,0.5335,19,SLC,90,0.3896,False


all designs


topType,family,SLC,TR,CVT,TW,MR,DS,HB,PFV,PTC,RC,SRW,TB,members
0,P1,90,26,40,0,0,0,0,0,11,0,1,6,174
1,P2,127,50,56,0,0,3,0,0,16,0,2,10,264
2,P3,101,31,37,0,0,1,0,0,11,0,1,10,192
3,P4,0,1,2,0,10,0,0,0,1,0,0,0,14
4,P5,127,50,67,28,0,3,0,0,16,0,2,10,303


actively protected


topType,family,SLC,TR,CVT,TW,MR,DS,HB,PFV,PTC,RC,SRW,TB,members
0,P1,69,21,30,0,0,0,0,0,10,0,1,4,135
1,P2,90,42,41,0,0,3,0,0,15,0,2,6,199
2,P3,76,25,28,0,0,1,0,0,11,0,1,6,148
3,P4,0,1,2,0,8,0,0,0,1,0,0,0,12
4,P5,90,42,50,23,0,3,0,0,15,0,2,6,231


value,<= 2011,2012-15,2016-19,2020-23,2024-26 (partial)
family,,,,,
P1,11,7,51,82,23
P2,19,13,73,125,34
P3,12,9,56,91,24
P4,1,0,8,5,0
P5,20,15,91,140,37


value,BoxTail,Conventional,Cruciform,Fins,H-Tail,HorizStab,Inv_V-Tail,Oth,T-Tail,Tailless,V-Tail,VertFin
family,,,,,,,,,,,,
P1,12,21,1,17,9,4,3,6,21,36,31,13
P2,13,36,2,27,11,7,3,8,31,64,39,23
P3,10,26,1,21,8,5,3,8,18,41,32,19
P4,1,2,0,1,0,0,1,0,0,9,0,0
P5,14,41,3,28,19,9,4,8,33,76,42,26


## 6.3b Premise check

Per cohort and base, 200 random subsets of the cohort's designs of the family's size (seed 42): cohesion = mean w15 Gower distance within the family against the band of the subsets; separation = mean distance family-to-rest against the same band. The premise holds when cohesion is below the band AND separation is above it. Then Q of each complete cohort (all designs) split into within-family, within-rest and between pairs, to see which pairs carry the ΔQ of Task 1.

In [5]:
pr = ff.premise_check(ctx)
display(pr[pr['cohort'].eq('2020-23')].drop(columns=['partial']))
display(pr[pr['read'].astype(bool)][['family', 'base', 'cohort', 'family designs', 'cohesion', 'cohesion band low', 'separation', 'separation band high', 'premise holds']])
qd = ff.q_decomposition(ctx)
display(qd[qd['cohort'].eq('2020-23')])

,family,base,cohort,designs in cohort,family designs,rest designs,read,cohesion,cohesion band low,cohesion band high,cohesion below band,separation,separation band low,separation band high,separation above band,premise holds
3,P1,all designs,2020-23,237,82,155,True,0.3986,0.4965,0.5432,True,0.5365,0.5154,0.5274,True,True
8,P1,actively protected,2020-23,192,71,121,True,0.4059,0.4878,0.5444,True,0.5343,0.5123,0.5248,True,True
13,P2,all designs,2020-23,237,125,112,True,0.4465,0.5036,0.5380,True,0.5534,0.5193,0.5231,True,True
18,P2,actively protected,2020-23,192,109,83,True,0.4499,0.4998,0.5344,True,0.5534,0.5151,0.5224,True,True
23,P3,all designs,2020-23,237,91,146,True,0.4242,0.4970,0.5435,True,0.5365,0.5168,0.5253,True,True
28,P3,actively protected,2020-23,192,78,114,True,0.4272,0.4970,0.5425,True,0.5341,0.5145,0.5239,True,True
33,P4,all designs,2020-23,237,5,232,False,NaN,NaN,NaN,None,NaN,NaN,NaN,None,None
38,P4,actively protected,2020-23,192,4,188,False,NaN,NaN,NaN,None,NaN,NaN,NaN,None,None
43,P5,all designs,2020-23,237,140,97,True,0.4567,0.5099,0.5346,True,0.5592,0.5174,0.5241,True,True
48,P5,actively protected,2020-23,192,123,69,True,0.4596,0.5039,0.5340,True,0.5616,0.5121,0.5242,True,True


,family,base,cohort,family designs,cohesion,cohesion band low,separation,separation band high,premise holds
0,P1,all designs,<= 2011,11,0.3803,0.4809,0.5950,0.5981,False
2,P1,all designs,2016-19,51,0.4631,0.5369,0.5891,0.5828,True
3,P1,all designs,2020-23,82,0.3986,0.4965,0.5365,0.5274,True
4,P1,all designs,2024-26 (partial),23,0.4321,0.5085,0.5754,0.5710,True
7,P1,actively protected,2016-19,35,0.4355,0.5401,0.6003,0.5890,True
8,P1,actively protected,2020-23,71,0.4059,0.4878,0.5343,0.5248,True
9,P1,actively protected,2024-26 (partial),23,0.4321,0.5068,0.5721,0.5615,True
10,P2,all designs,<= 2011,19,0.4242,0.5216,0.6054,0.5864,True
11,P2,all designs,2012-15,13,0.4296,0.4562,0.5765,0.5734,True
12,P2,all designs,2016-19,73,0.4707,0.5473,0.5929,0.5766,True


,family,cohort,designs,family designs,block,pairs,share of pairs,mean distance,contribution to Q,Q of cohort,"contribution, mean of the two earliest cohorts",change 2020-23
9,P1,2020-23,237,82,within family,6724,0.1197,0.3937,0.0471,0.5185,0.0072,0.0399
10,P1,2020-23,237,82,within rest,24025,0.4277,0.5343,0.2286,0.5185,0.3895,-0.1609
11,P1,2020-23,237,82,between,25420,0.4526,0.5365,0.2428,0.5185,0.1497,0.0931
21,P2,2020-23,237,125,within family,15625,0.2782,0.4429,0.1232,0.5185,0.0283,0.0949
22,P2,2020-23,237,125,within rest,12544,0.2233,0.5345,0.1194,0.5185,0.2903,-0.1709
23,P2,2020-23,237,125,between,28000,0.4985,0.5534,0.2759,0.5185,0.2278,0.0481
33,P3,2020-23,237,91,within family,8281,0.1474,0.4195,0.0618,0.5185,0.0103,0.0515
34,P3,2020-23,237,91,within rest,21316,0.3795,0.5345,0.2028,0.5185,0.3699,-0.1671
35,P3,2020-23,237,91,between,26572,0.4731,0.5365,0.2538,0.5185,0.1662,0.0876
45,P4,2020-23,237,5,within family,25,0.0004,0.4121,0.0002,0.5185,0.0000,0.0002


## 6.3c Condition 1

The family's share per priority year, read years only (20 designs or more), met above 50 % in two consecutive years; the same code path as `q7_condition1_family`. Three consecutive years as sensitivity.

In [6]:
c1 = ff.condition1(ctx)
y = c1['yearly']
display(y[y['read'].astype(bool)].pivot(index='priority year', columns=['base', 'family'], values='family share'))
for k, m in c1['met'].items():
    print(k, '| two consecutive years:', m['two'] or 'none', '| three:', m['three'] or 'none')

base,all designs,actively protected,all designs,actively protected,all designs,actively protected,all designs,actively protected,all designs,actively protected
family,P1,P1,P2,P2,P3,P3,P4,P4,P5,P5
priority year,,,,,,,,,,
2015,0.2857,NaN,0.3810,NaN,0.3333,NaN,0.0000,NaN,0.3810,NaN
2016,0.1765,0.1667,0.2353,0.2083,0.2059,0.1667,0.0588,0.0833,0.3235,0.2917
2017,0.1346,0.1515,0.1923,0.2424,0.1538,0.1818,0.0577,0.0909,0.3077,0.3939
2018,0.3030,0.2973,0.4394,0.3784,0.3333,0.3243,0.0152,0.0270,0.4697,0.4054
2019,0.3051,0.3409,0.4407,0.4545,0.3220,0.3409,0.0339,0.0455,0.5593,0.5909
2020,0.2464,0.2963,0.4203,0.4815,0.2609,0.2963,0.0145,0.0185,0.4638,0.5370
2021,0.3750,0.3729,0.5417,0.5593,0.3889,0.3898,0.0278,0.0339,0.6389,0.6780
2022,0.4167,0.4286,0.6458,0.7143,0.5000,0.5429,0.0208,0.0000,0.6875,0.7429


('P1', 'all designs') | two consecutive years: none | three: none
('P1', 'actively protected') | two consecutive years: none | three: none
('P2', 'all designs') | two consecutive years: [2021, 2022, 2023] | three: [2021, 2022, 2023]
('P2', 'actively protected') | two consecutive years: [2021, 2022, 2023] | three: [2021, 2022, 2023]
('P3', 'all designs') | two consecutive years: none | three: none
('P3', 'actively protected') | two consecutive years: none | three: none
('P4', 'all designs') | two consecutive years: none | three: none
('P4', 'actively protected') | two consecutive years: none | three: none
('P5', 'all designs') | two consecutive years: [2021, 2022, 2023] | three: [2021, 2022, 2023]
('P5', 'actively protected') | two consecutive years: [2019, 2020, 2021, 2022, 2023] | three: [2019, 2020, 2021, 2022, 2023]


## 6.3d Condition 2

²D rarefied to 40 against the 200-permutation band (seed 42, fresh generator per family and base) on the partition {family} ∪ {each remaining architecture as its own option}, exactly as F was computed.

In [7]:
c2 = ff.condition2(ctx)
display(c2[~c2['partial']][['family', 'base', 'partition', 'cohort', 'designs', 'top option', 'top share', 'D2', 'D2 permutation low', 'D2 permutation high', 'below band']])

,family,base,partition,cohort,designs,top option,top share,D2,D2 permutation low,D2 permutation high,below band
0,P1,all designs,{P1} ∪ {each remaining architecture},<= 2011,63,TR,0.286,5.82,4.49,7.18,False
1,P1,all designs,{P1} ∪ {each remaining architecture},2012-15,58,TR,0.328,5.52,4.44,7.03,False
2,P1,all designs,{P1} ∪ {each remaining architecture},2016-19,211,"P1 fixed wing, boom-borne distributed propulsion",0.242,6.19,5.21,6.33,False
3,P1,all designs,{P1} ∪ {each remaining architecture},2020-23,237,"P1 fixed wing, boom-borne distributed propulsion",0.346,4.87,5.07,6.32,True
5,P1,actively protected,{P1} ∪ {each remaining architecture},<= 2011,18,TR,0.444,3.31,2.79,7.05,False
6,P1,actively protected,{P1} ∪ {each remaining architecture},2012-15,33,TR,0.424,4.14,3.44,6.76,False
7,P1,actively protected,{P1} ∪ {each remaining architecture},2016-19,138,"P1 fixed wing, boom-borne distributed propulsion",0.254,5.95,4.52,6.12,False
8,P1,actively protected,{P1} ∪ {each remaining architecture},2020-23,192,"P1 fixed wing, boom-borne distributed propulsion",0.370,4.60,4.70,5.87,True
10,P2,all designs,{P2} ∪ {each remaining architecture},<= 2011,63,"P2 fixed wing, distributed propulsion",0.302,5.08,3.21,5.49,False
11,P2,all designs,{P2} ∪ {each remaining architecture},2012-15,58,TR,0.293,5.44,2.83,5.40,False


## 6.3e Condition 3

Quoted from Task 1 per base (w15, complete cohorts): Rao's Q is cohort-wide and blind to the family, so it is never recomputed per family. This step loads the Task 1 cache (a few minutes).

In [8]:
c3 = ff.condition3(ctx)
display(c3)

,base,level,weighting,ΔQ 2020-23,ΔQ lower band edge 2020-23,below band 2020-23,cohorts met,family-independent
0,all designs,A,w15,-0.0279,-0.0190,True,2020-23,True
1,all designs,A·W·T,w15,-0.0299,-0.0204,True,2020-23,True
2,all designs,A·U,w15,-0.0299,-0.0168,True,2020-23,True
3,all designs,F,w15,-0.0279,-0.0190,True,2020-23,True
4,actively protected,A,w15,0.0091,-0.0221,False,none,True
5,actively protected,A·W·T,w15,0.0064,-0.0226,False,none,True
6,actively protected,A·U,w15,0.0064,-0.0177,False,none,True
7,actively protected,F,w15,0.0091,-0.0221,False,none,True


## 6.4 Verdict

The shape of `q7_verdict_table.csv` with the cohesion and separation rows; "Met" only when all three hold in the same period on the same base.

In [9]:
vt = ff.verdict_table(ctx)
display(vt)
for base, _ in ff.BASES:
    print(base); display(ff.verdict_md_table(ctx, base))

,base,condition,P1,P2,P3,P4,P5
0,all designs,premise: cohesion 2020-23,0.3986 (band 0.4965 to 0.5432) ✓ below,0.4465 (band 0.5036 to 0.5380) ✓ below,0.4242 (band 0.4970 to 0.5435) ✓ below,not read,0.4567 (band 0.5099 to 0.5346) ✓ below
1,all designs,premise: separation 2020-23,0.5365 (band 0.5154 to 0.5274) ✓ above,0.5534 (band 0.5193 to 0.5231) ✓ above,0.5365 (band 0.5168 to 0.5253) ✓ above,not read,0.5592 (band 0.5174 to 0.5241) ✓ above
2,all designs,premise: cohesion below AND separation above the band (complete cohorts),"✓ holds in 2016-19, 2020-23","✓ holds in <= 2011, 2012-15, 2016-19, 2020-23","✓ holds in 2016-19, 2020-23",✗ does not hold,"✓ holds in <= 2011, 2016-19, 2020-23"
3,all designs,"1, counts: the family above 50 % of the designs, two consecutive priority years","✗ not met, at most 41.7 % (2022)","✓ met in 2021, 2022 and 2023 (54.2 %, 64.6 %, 54.2 %)","✗ not met, at most 50.0 % (2022)","✗ not met, at most 5.9 % (2016)","✓ met in 2021, 2022 and 2023 (63.9 %, 68.8 %, 60.4 %)"
4,all designs,"1, sensitivity: three consecutive priority years",✗ not met,"✓ 2021, 2022, 2023",✗ not met,✗ not met,"✓ 2021, 2022, 2023"
5,all designs,"2, balance: ²D below the permutation band, partition {family} ∪ {each remaining architecture}",✓ met in 2020-23,✓ met in 2020-23,✓ met in 2020-23,✓ met in 2020-23,✓ met in 2020-23
6,all designs,"3, form: Q below the two earliest cohorts by more than the band (Task 1, level A; family-independent)",✓ met in 2020-23,✓ met in 2020-23,✓ met in 2020-23,✓ met in 2020-23,✓ met in 2020-23
7,all designs,all three in the same period,✗ not met,✓ met in 2020-23,✗ not met,✗ not met,✓ met in 2020-23
8,actively protected,premise: cohesion 2020-23,0.4059 (band 0.4878 to 0.5444) ✓ below,0.4499 (band 0.4998 to 0.5344) ✓ below,0.4272 (band 0.4970 to 0.5425) ✓ below,not read,0.4596 (band 0.5039 to 0.5340) ✓ below
9,actively protected,premise: separation 2020-23,0.5343 (band 0.5123 to 0.5248) ✓ above,0.5534 (band 0.5151 to 0.5224) ✓ above,0.5341 (band 0.5145 to 0.5239) ✓ above,not read,0.5616 (band 0.5121 to 0.5242) ✓ above


all designs


,family,premise,condition 1: largest share (year); years met,condition 2: cohorts met,"condition 3 (cohort verdict, family-independent)","all three, same period"
0,P1,"✓ holds in 2016-19, 2020-23",41.7 % (2022); none,2020-23,✓ met in 2020-23,✗ not met
1,P2,"✓ holds in <= 2011, 2012-15, 2016-19, 2020-23","64.6 % (2022); 2021, 2022, 2023",2020-23,✓ met in 2020-23,✓ met in 2020-23
2,P3,"✓ holds in 2016-19, 2020-23",50.0 % (2022); none,2020-23,✓ met in 2020-23,✗ not met
3,P4,✗ does not hold,5.9 % (2016); none,2020-23,✓ met in 2020-23,✗ not met
4,P5,"✓ holds in <= 2011, 2016-19, 2020-23","68.8 % (2022); 2021, 2022, 2023",2020-23,✓ met in 2020-23,✓ met in 2020-23


actively protected


,family,premise,condition 1: largest share (year); years met,condition 2: cohorts met,"condition 3 (cohort verdict, family-independent)","all three, same period"
0,P1,"✓ holds in 2016-19, 2020-23",42.9 % (2022); none,2020-23,✗ not met,✗ not met
1,P2,"✓ holds in 2016-19, 2020-23","71.4 % (2022); 2021, 2022, 2023",2020-23,✗ not met,✗ not met
2,P3,"✓ holds in 2016-19, 2020-23",54.3 % (2022); none,2020-23,✗ not met,✗ not met
3,P4,✗ does not hold,9.1 % (2017); none,2020-23,✗ not met,✗ not met
4,P5,"✓ holds in 2016-19, 2020-23","74.3 % (2022); 2019, 2020, 2021, 2022, 2023",2020-23,✗ not met,✗ not met


## Write the outputs

Results folder, `LaTeX_v15/Data_for_writing/csv/` (+ the md in `Data_for_writing/`) and `handoff_2026-10-16/task6/` (every output plus `viz/thesis_form_families.py` and this notebook).

In [10]:
written = ff.write_outputs(ctx, 'task6')
print('\n'.join(map(str, written)))
display(Markdown((OUT / 'NUMBERS_VI_form_families.md').read_text(encoding='utf-8')))

/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_analysis/labelling_analysis_thesis/thesis_figures_v2/Part_III_dominant_design/III3_form_families/exploratory/q8_form_variables.csv
/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_analysis/labelling_analysis_thesis/LaTeX_v15/Data_for_writing/csv/q8_form_variables.csv
/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_analysis/labelling_analysis_thesis/thesis_figures_v2/Part_III_dominant_design/III3_form_families/exploratory/q8_families_membership.csv
/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_analysis/labelling_analysis_thesis/LaTeX_v15/Data_for_writing/csv/q8_families_membership.csv
/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_an

# NUMBERS_VI — form-defined families (Task 6, exploratory)

Written by notebook 55 (`viz/thesis_form_families.py`); every value is recomputed when the notebook runs. CSVs are in `Data_for_writing/csv/` (`q8_form_variables.csv`, `q8_families_membership.csv`, `q8_families_premise.csv`, `q8_families_q_decomposition.csv`, `q8_families_condition1.csv`, `q8_families_condition2.csv`, `q8_families_condition3.csv`, `q8_families_verdict.csv`).

Exploratory, after Task 1, on the w15 weighting only. Nothing here changes F or any number of the dissertation (F stays as decided on 2026-10-14). The question: is there a family defined by form (wing, tilt, booms, propulsor count, enclosure), not by architecture, that is coherent in form AND meets share dominance (condition 1) and balance reduction (condition 2)? Condition 3 (Rao's Q) is cohort-wide and blind to the family, so it is never recomputed per family: the Task 1 cohort verdict is quoted per base. "Met" only when all three hold in the same period on the same base.

## Declared families (fixed before running; none added or tuned after seeing results)

| id   | name                                             | rule                                                            | premise                                                                                   |
|:-----|:-------------------------------------------------|:----------------------------------------------------------------|:------------------------------------------------------------------------------------------|
| P1   | fixed wing, boom-borne distributed propulsion    | wing_n >= 1 AND wing_tilts = no AND boom_n >= 1 AND band >= 5-6 | F in form terms: a lifting wing for range, many small propulsors on booms; propulsor tilt free |
| P2   | fixed wing, distributed propulsion               | wing_n >= 1 AND wing_tilts = no AND band >= 5-6                 | P1 without the boom rule                                                                  |
| P3   | fixed wing, open distributed propulsion          | P2 AND enclosure = not ducted                                   | ducted propulsors are a distinct form                                                     |
| P4   | wingless multicopter, distributed                | wing_n = 0 AND band >= 5-6 AND enclosure = not ducted           | the Multicopter counterpart of P3                                                         |
| P5   | fixed wing with distributed propulsion, any tilt | wing_n >= 1 AND band >= 5-6                                     | upper bound, includes Tilt Wing and Tilt Frame                                            |

Rule variables: wing_n (wCount, 0 when no wing), wing_tilts (any wing panel Tilt), boom_n (boom1..6_count summed, 0 without booms), band (propulsor band 1-3, 4, 5-6, 7-8, 9+), enclosure (of the set with most propulsors). Empennage type is never a membership rule. A design whose rule variable is blank (hidden by an override, or no propulsor card / count) is not a member.

## Membership (size, largest architecture, renamed?)

Source: `q8_families_membership.csv` (summary rows) and the crosstabs below.

| family   | base               |   designs in base |   members |   share of base |   not members for a blank rule variable | top architecture   |   top architecture share | renamed architecture (> 85 %)   |
|:---------|:-------------------|------------------:|----------:|----------------:|----------------------------------------:|:-------------------|-------------------------:|:--------------------------------|
| P1       | all designs        |               625 |       174 |          0.2784 |                                      30 | SLC                |                   0.5172 | False                           |
| P1       | actively protected |               433 |       135 |          0.3118 |                                      19 | SLC                |                   0.5111 | False                           |
| P2       | all designs        |               625 |       264 |          0.4224 |                                      30 | SLC                |                   0.4811 | False                           |
| P2       | actively protected |               433 |       199 |          0.4596 |                                      19 | SLC                |                   0.4523 | False                           |
| P3       | all designs        |               625 |       192 |          0.3072 |                                      31 | SLC                |                   0.526  | False                           |
| P3       | actively protected |               433 |       148 |          0.3418 |                                      20 | SLC                |                   0.5135 | False                           |
| P4       | all designs        |               625 |        14 |          0.0224 |                                      31 | MR                 |                   0.7143 | False                           |
| P4       | actively protected |               433 |        12 |          0.0277 |                                      20 | MR                 |                   0.6667 | False                           |
| P5       | all designs        |               625 |       303 |          0.4848 |                                      30 | SLC                |                   0.4191 | False                           |
| P5       | actively protected |               433 |       231 |          0.5335 |                                      19 | SLC                |                   0.3896 | False                           |

Family x architecture, all designs (codes):

| family   |   SLC |   TR |   CVT |   TW |   MR |   DS |   HB |   PFV |   PTC |   RC |   SRW |   TB |   members |
|:---------|------:|-----:|------:|-----:|-----:|-----:|-----:|------:|------:|-----:|------:|-----:|----------:|
| P1       |    90 |   26 |    40 |    0 |    0 |    0 |    0 |     0 |    11 |    0 |     1 |    6 |       174 |
| P2       |   127 |   50 |    56 |    0 |    0 |    3 |    0 |     0 |    16 |    0 |     2 |   10 |       264 |
| P3       |   101 |   31 |    37 |    0 |    0 |    1 |    0 |     0 |    11 |    0 |     1 |   10 |       192 |
| P4       |     0 |    1 |     2 |    0 |   10 |    0 |    0 |     0 |     1 |    0 |     0 |    0 |        14 |
| P5       |   127 |   50 |    67 |   28 |    0 |    3 |    0 |     0 |    16 |    0 |     2 |   10 |       303 |

Family x architecture, actively protected (codes):

| family   |   SLC |   TR |   CVT |   TW |   MR |   DS |   HB |   PFV |   PTC |   RC |   SRW |   TB |   members |
|:---------|------:|-----:|------:|-----:|-----:|-----:|-----:|------:|------:|-----:|------:|-----:|----------:|
| P1       |    69 |   21 |    30 |    0 |    0 |    0 |    0 |     0 |    10 |    0 |     1 |    4 |       135 |
| P2       |    90 |   42 |    41 |    0 |    0 |    3 |    0 |     0 |    15 |    0 |     2 |    6 |       199 |
| P3       |    76 |   25 |    28 |    0 |    0 |    1 |    0 |     0 |    11 |    0 |     1 |    6 |       148 |
| P4       |     0 |    1 |     2 |    0 |    8 |    0 |    0 |     0 |     1 |    0 |     0 |    0 |        12 |
| P5       |    90 |   42 |    50 |   23 |    0 |    3 |    0 |     0 |    15 |    0 |     2 |    6 |       231 |

## Verdict table, all designs (main)

Source: `q8_families_verdict.csv`. Premise = cohesion below AND separation above the band of 200 random subsets of the cohort of the family's size (seed 42); condition 1 read years only (20 designs or more), two consecutive years above 50 %; condition 2 ²D rarefied to 40 against the 200-permutation band on the partition {family} ∪ {each remaining architecture as its own option}, complete cohorts; condition 3 = Task 1, level A, w15, family-independent.

| family   | premise                                       | condition 1: largest share (year); years met   | condition 2: cohorts met   | condition 3 (cohort verdict, family-independent)   | all three, same period   |
|:---------|:----------------------------------------------|:-----------------------------------------------|:---------------------------|:---------------------------------------------------|:-------------------------|
| P1       | ✓ holds in 2016-19, 2020-23                   | 41.7 % (2022); none                            | 2020-23                    | ✓ met in 2020-23                                   | ✗ not met                |
| P2       | ✓ holds in <= 2011, 2012-15, 2016-19, 2020-23 | 64.6 % (2022); 2021, 2022, 2023                | 2020-23                    | ✓ met in 2020-23                                   | ✓ met in 2020-23         |
| P3       | ✓ holds in 2016-19, 2020-23                   | 50.0 % (2022); none                            | 2020-23                    | ✓ met in 2020-23                                   | ✗ not met                |
| P4       | ✗ does not hold                               | 5.9 % (2016); none                             | 2020-23                    | ✓ met in 2020-23                                   | ✗ not met                |
| P5       | ✓ holds in <= 2011, 2016-19, 2020-23          | 68.8 % (2022); 2021, 2022, 2023                | 2020-23                    | ✓ met in 2020-23                                   | ✓ met in 2020-23         |

**P1, fixed wing, boom-borne distributed propulsion** (wing_n >= 1 AND wing_tilts = no AND boom_n >= 1 AND band >= 5-6). On all designs it joins 174 designs (27.8 % of 625): Separate Lift + Cruise 90, Combined Vectored Thrust 40, Tilt Propulsor 26, Pitch-to-Cruise 11, Tilt Frame 6, Stopped/Slowed Rotor Wing 1; 30 designs are not members because a rule variable is blank. Its largest architecture is Separate Lift + Cruise at 51.7 %, so it is not one architecture renamed. Empennage inside the family: Tailless 36, V-Tail 31, T-Tail 21, Conventional 21. In 2020-23 its cohesion is 0.3986 against a band of 0.4965 to 0.5432 (below) and its separation 0.5365 against 0.5154 to 0.5274 (above); the premise holds in 2016-19, 2020-23. Condition 1 is not met: its largest share of a read year is 41.7 % in 2022 (2016: 17.6 %, 12 more designs needed; 2017: 13.5 %, 20 more designs needed; 2018: 30.3 %, 14 more designs needed; 2019: 30.5 %, 12 more designs needed; 2020: 24.6 %, 18 more designs needed; 2021: 37.5 %, 10 more designs needed; 2022: 41.7 %, 5 more designs needed; 2023: 37.5 %, 7 more designs needed). Condition 2 is met in 2020-23 on the partition {P1} ∪ {each remaining architecture}. Condition 3 is the cohort verdict of Task 1 on this base, independent of the family: ΔQ 2020-23 -0.0279 against a lower band edge of -0.0190, met. The three conditions do not hold in the same period, so the family is not a dominant design on this base.

**P2, fixed wing, distributed propulsion** (wing_n >= 1 AND wing_tilts = no AND band >= 5-6). On all designs it joins 264 designs (42.2 % of 625): Separate Lift + Cruise 127, Combined Vectored Thrust 56, Tilt Propulsor 50, Pitch-to-Cruise 16, Tilt Frame 10, Deflected Slipstream 3, Stopped/Slowed Rotor Wing 2; 30 designs are not members because a rule variable is blank. Its largest architecture is Separate Lift + Cruise at 48.1 %, so it is not one architecture renamed. Empennage inside the family: Tailless 64, V-Tail 39, Conventional 36, T-Tail 31. In 2020-23 its cohesion is 0.4465 against a band of 0.5036 to 0.5380 (below) and its separation 0.5534 against 0.5193 to 0.5231 (above); the premise holds in <= 2011, 2012-15, 2016-19, 2020-23. Condition 1 is met in 2021, 2022 and 2023 (54.2 %: 39 of 72; 64.6 %: 31 of 48; 54.2 %: 26 of 48), and with three consecutive years in 2021, 2022, 2023. Condition 2 is met in 2020-23 on the partition {P2} ∪ {each remaining architecture}. Condition 3 is the cohort verdict of Task 1 on this base, independent of the family: ΔQ 2020-23 -0.0279 against a lower band edge of -0.0190, met. All three hold in the same period: 2020-23.

**P3, fixed wing, open distributed propulsion** (P2 AND enclosure = not ducted). On all designs it joins 192 designs (30.7 % of 625): Separate Lift + Cruise 101, Combined Vectored Thrust 37, Tilt Propulsor 31, Pitch-to-Cruise 11, Tilt Frame 10, Deflected Slipstream 1, Stopped/Slowed Rotor Wing 1; 31 designs are not members because a rule variable is blank. Its largest architecture is Separate Lift + Cruise at 52.6 %, so it is not one architecture renamed. Empennage inside the family: Tailless 41, V-Tail 32, Conventional 26, Fins 21. In 2020-23 its cohesion is 0.4242 against a band of 0.4970 to 0.5435 (below) and its separation 0.5365 against 0.5168 to 0.5253 (above); the premise holds in 2016-19, 2020-23. Condition 1 is not met: its largest share of a read year is 50.0 % in 2022 (2016: 20.6 %, 11 more designs needed; 2017: 15.4 %, 19 more designs needed; 2018: 33.3 %, 12 more designs needed; 2019: 32.2 %, 11 more designs needed; 2020: 26.1 %, 17 more designs needed; 2021: 38.9 %, 9 more designs needed; 2022: 50.0 %, 1 more design needed; 2023: 43.8 %, 4 more designs needed). Condition 2 is met in 2020-23 on the partition {P3} ∪ {each remaining architecture}. Condition 3 is the cohort verdict of Task 1 on this base, independent of the family: ΔQ 2020-23 -0.0279 against a lower band edge of -0.0190, met. The three conditions do not hold in the same period, so the family is not a dominant design on this base.

**P4, wingless multicopter, distributed** (wing_n = 0 AND band >= 5-6 AND enclosure = not ducted). On all designs it joins 14 designs (2.2 % of 625): Multicopter 10, Combined Vectored Thrust 2, Pitch-to-Cruise 1, Tilt Propulsor 1; 31 designs are not members because a rule variable is blank. Its largest architecture is Multicopter at 71.4 %, so it is not one architecture renamed. Empennage inside the family: Tailless 9, Conventional 2, Fins 1, BoxTail 1. It has too few designs in 2020-23 for the premise check. Condition 1 is not met: its largest share of a read year is 5.9 % in 2016 (2016: 5.9 %, 16 more designs needed; 2017: 5.8 %, 24 more designs needed; 2018: 1.5 %, 33 more designs needed; 2019: 3.4 %, 28 more designs needed; 2020: 1.5 %, 34 more designs needed; 2021: 2.8 %, 35 more designs needed; 2022: 2.1 %, 24 more designs needed; 2023: 2.1 %, 24 more designs needed). Condition 2 is met in 2020-23 on the partition {P4} ∪ {each remaining architecture}. Condition 3 is the cohort verdict of Task 1 on this base, independent of the family: ΔQ 2020-23 -0.0279 against a lower band edge of -0.0190, met. The three conditions do not hold in the same period, so the family is not a dominant design on this base.

**P5, fixed wing with distributed propulsion, any tilt** (wing_n >= 1 AND band >= 5-6). On all designs it joins 303 designs (48.5 % of 625): Separate Lift + Cruise 127, Combined Vectored Thrust 67, Tilt Propulsor 50, Tilt Wing 28, Pitch-to-Cruise 16, Tilt Frame 10, Deflected Slipstream 3, Stopped/Slowed Rotor Wing 2; 30 designs are not members because a rule variable is blank. Its largest architecture is Separate Lift + Cruise at 41.9 %, so it is not one architecture renamed. Empennage inside the family: Tailless 76, V-Tail 42, Conventional 41, T-Tail 33. In 2020-23 its cohesion is 0.4567 against a band of 0.5099 to 0.5346 (below) and its separation 0.5592 against 0.5174 to 0.5241 (above); the premise holds in <= 2011, 2016-19, 2020-23. Condition 1 is met in 2021, 2022 and 2023 (63.9 %: 46 of 72; 68.8 %: 33 of 48; 60.4 %: 29 of 48), and with three consecutive years in 2021, 2022, 2023. Condition 2 is met in 2020-23 on the partition {P5} ∪ {each remaining architecture}. Condition 3 is the cohort verdict of Task 1 on this base, independent of the family: ΔQ 2020-23 -0.0279 against a lower band edge of -0.0190, met. All three hold in the same period: 2020-23.

## Verdict table, actively protected (sensitivity)

Source: `q8_families_verdict.csv`. Premise = cohesion below AND separation above the band of 200 random subsets of the cohort of the family's size (seed 42); condition 1 read years only (20 designs or more), two consecutive years above 50 %; condition 2 ²D rarefied to 40 against the 200-permutation band on the partition {family} ∪ {each remaining architecture as its own option}, complete cohorts; condition 3 = Task 1, level A, w15, family-independent.

| family   | premise                     | condition 1: largest share (year); years met   | condition 2: cohorts met   | condition 3 (cohort verdict, family-independent)   | all three, same period   |
|:---------|:----------------------------|:-----------------------------------------------|:---------------------------|:---------------------------------------------------|:-------------------------|
| P1       | ✓ holds in 2016-19, 2020-23 | 42.9 % (2022); none                            | 2020-23                    | ✗ not met                                          | ✗ not met                |
| P2       | ✓ holds in 2016-19, 2020-23 | 71.4 % (2022); 2021, 2022, 2023                | 2020-23                    | ✗ not met                                          | ✗ not met                |
| P3       | ✓ holds in 2016-19, 2020-23 | 54.3 % (2022); none                            | 2020-23                    | ✗ not met                                          | ✗ not met                |
| P4       | ✗ does not hold             | 9.1 % (2017); none                             | 2020-23                    | ✗ not met                                          | ✗ not met                |
| P5       | ✓ holds in 2016-19, 2020-23 | 74.3 % (2022); 2019, 2020, 2021, 2022, 2023    | 2020-23                    | ✗ not met                                          | ✗ not met                |

**P1, fixed wing, boom-borne distributed propulsion** (wing_n >= 1 AND wing_tilts = no AND boom_n >= 1 AND band >= 5-6). On actively protected it joins 135 designs (31.2 % of 433): Separate Lift + Cruise 69, Combined Vectored Thrust 30, Tilt Propulsor 21, Pitch-to-Cruise 10, Tilt Frame 4, Stopped/Slowed Rotor Wing 1; 19 designs are not members because a rule variable is blank. Its largest architecture is Separate Lift + Cruise at 51.1 %, so it is not one architecture renamed. Empennage inside the family: Tailless 29, V-Tail 24, T-Tail 18, Fins 16. In 2020-23 its cohesion is 0.4059 against a band of 0.4878 to 0.5444 (below) and its separation 0.5343 against 0.5123 to 0.5248 (above); the premise holds in 2016-19, 2020-23. Condition 1 is not met: its largest share of a read year is 42.9 % in 2022 (2016: 16.7 %, 9 more designs needed; 2017: 15.2 %, 12 more designs needed; 2018: 29.7 %, 8 more designs needed; 2019: 34.1 %, 8 more designs needed; 2020: 29.6 %, 12 more designs needed; 2021: 37.3 %, 8 more designs needed; 2022: 42.9 %, 3 more designs needed; 2023: 40.9 %, 5 more designs needed). Condition 2 is met in 2020-23 on the partition {P1} ∪ {each remaining architecture}. Condition 3 is the cohort verdict of Task 1 on this base, independent of the family: ΔQ 2020-23 +0.0091 against a lower band edge of -0.0221, not met. The three conditions do not hold in the same period, so the family is not a dominant design on this base.

**P2, fixed wing, distributed propulsion** (wing_n >= 1 AND wing_tilts = no AND band >= 5-6). On actively protected it joins 199 designs (46.0 % of 433): Separate Lift + Cruise 90, Tilt Propulsor 42, Combined Vectored Thrust 41, Pitch-to-Cruise 15, Tilt Frame 6, Deflected Slipstream 3, Stopped/Slowed Rotor Wing 2; 19 designs are not members because a rule variable is blank. Its largest architecture is Separate Lift + Cruise at 45.2 %, so it is not one architecture renamed. Empennage inside the family: Tailless 48, V-Tail 32, Fins 26, T-Tail 24. In 2020-23 its cohesion is 0.4499 against a band of 0.4998 to 0.5344 (below) and its separation 0.5534 against 0.5151 to 0.5224 (above); the premise holds in 2016-19, 2020-23. Condition 1 is met in 2021, 2022 and 2023 (55.9 %: 33 of 59; 71.4 %: 25 of 35; 56.8 %: 25 of 44), and with three consecutive years in 2021, 2022, 2023. Condition 2 is met in 2020-23 on the partition {P2} ∪ {each remaining architecture}. Condition 3 is the cohort verdict of Task 1 on this base, independent of the family: ΔQ 2020-23 +0.0091 against a lower band edge of -0.0221, not met. The three conditions do not hold in the same period, so the family is not a dominant design on this base.

**P3, fixed wing, open distributed propulsion** (P2 AND enclosure = not ducted). On actively protected it joins 148 designs (34.2 % of 433): Separate Lift + Cruise 76, Combined Vectored Thrust 28, Tilt Propulsor 25, Pitch-to-Cruise 11, Tilt Frame 6, Deflected Slipstream 1, Stopped/Slowed Rotor Wing 1; 20 designs are not members because a rule variable is blank. Its largest architecture is Separate Lift + Cruise at 51.3 %, so it is not one architecture renamed. Empennage inside the family: Tailless 33, V-Tail 25, Fins 20, T-Tail 15. In 2020-23 its cohesion is 0.4272 against a band of 0.4970 to 0.5425 (below) and its separation 0.5341 against 0.5145 to 0.5239 (above); the premise holds in 2016-19, 2020-23. Condition 1 is not met: its largest share of a read year is 54.3 % in 2022 (2016: 16.7 %, 9 more designs needed; 2017: 18.2 %, 11 more designs needed; 2018: 32.4 %, 7 more designs needed; 2019: 34.1 %, 8 more designs needed; 2020: 29.6 %, 12 more designs needed; 2021: 39.0 %, 7 more designs needed; 2023: 45.5 %, 3 more designs needed). Condition 2 is met in 2020-23 on the partition {P3} ∪ {each remaining architecture}. Condition 3 is the cohort verdict of Task 1 on this base, independent of the family: ΔQ 2020-23 +0.0091 against a lower band edge of -0.0221, not met. The three conditions do not hold in the same period, so the family is not a dominant design on this base.

**P4, wingless multicopter, distributed** (wing_n = 0 AND band >= 5-6 AND enclosure = not ducted). On actively protected it joins 12 designs (2.8 % of 433): Multicopter 8, Combined Vectored Thrust 2, Pitch-to-Cruise 1, Tilt Propulsor 1; 20 designs are not members because a rule variable is blank. Its largest architecture is Multicopter at 66.7 %, so it is not one architecture renamed. Empennage inside the family: Tailless 8, Conventional 2, Fins 1, Inv_V-Tail 1. It has too few designs in 2020-23 for the premise check. Condition 1 is not met: its largest share of a read year is 9.1 % in 2017 (2016: 8.3 %, 11 more designs needed; 2017: 9.1 %, 14 more designs needed; 2018: 2.7 %, 18 more designs needed; 2019: 4.5 %, 21 more designs needed; 2020: 1.8 %, 27 more designs needed; 2021: 3.4 %, 28 more designs needed; 2022: 0.0 %, 18 more designs needed; 2023: 2.3 %, 22 more designs needed). Condition 2 is met in 2020-23 on the partition {P4} ∪ {each remaining architecture}. Condition 3 is the cohort verdict of Task 1 on this base, independent of the family: ΔQ 2020-23 +0.0091 against a lower band edge of -0.0221, not met. The three conditions do not hold in the same period, so the family is not a dominant design on this base.

**P5, fixed wing with distributed propulsion, any tilt** (wing_n >= 1 AND band >= 5-6). On actively protected it joins 231 designs (53.3 % of 433): Separate Lift + Cruise 90, Combined Vectored Thrust 50, Tilt Propulsor 42, Tilt Wing 23, Pitch-to-Cruise 15, Tilt Frame 6, Deflected Slipstream 3, Stopped/Slowed Rotor Wing 2; 19 designs are not members because a rule variable is blank. Its largest architecture is Separate Lift + Cruise at 39.0 %, so it is not one architecture renamed. Empennage inside the family: Tailless 58, V-Tail 34, T-Tail 26, Fins 26. In 2020-23 its cohesion is 0.4596 against a band of 0.5039 to 0.5340 (below) and its separation 0.5616 against 0.5121 to 0.5242 (above); the premise holds in 2016-19, 2020-23. Condition 1 is met in 2019, 2020, 2021, 2022 and 2023 (59.1 %: 26 of 44; 53.7 %: 29 of 54; 67.8 %: 40 of 59; 74.3 %: 26 of 35; 63.6 %: 28 of 44), and with three consecutive years in 2019, 2020, 2021, 2022, 2023. Condition 2 is met in 2020-23 on the partition {P5} ∪ {each remaining architecture}. Condition 3 is the cohort verdict of Task 1 on this base, independent of the family: ΔQ 2020-23 +0.0091 against a lower band edge of -0.0221, not met. The three conditions do not hold in the same period, so the family is not a dominant design on this base.

## Condition 3 as quoted (Task 1, w15, complete cohorts; family-independent)

Source: `q8_families_condition3.csv` (from `q7_condition3_cohesion.csv` / the verdict detail of Task 1).

| base               | level   | weighting   |   ΔQ 2020-23 |   ΔQ lower band edge 2020-23 | below band 2020-23   | cohorts met   | family-independent   |
|:-------------------|:--------|:------------|-------------:|-----------------------------:|:---------------------|:--------------|:---------------------|
| all designs        | A       | w15         |      -0.0279 |                      -0.019  | True                 | 2020-23       | True                 |
| all designs        | A·W·T   | w15         |      -0.0299 |                      -0.0204 | True                 | 2020-23       | True                 |
| all designs        | A·U     | w15         |      -0.0299 |                      -0.0168 | True                 | 2020-23       | True                 |
| all designs        | F       | w15         |      -0.0279 |                      -0.019  | True                 | 2020-23       | True                 |
| actively protected | A       | w15         |       0.0091 |                      -0.0221 | False                | none          | True                 |
| actively protected | A·W·T   | w15         |       0.0064 |                      -0.0226 | False                | none          | True                 |
| actively protected | A·U     | w15         |       0.0064 |                      -0.0177 | False                | none          | True                 |
| actively protected | F       | w15         |       0.0091 |                      -0.0221 | False                | none          | True                 |

## Which pairs carry ΔQ (all designs, w15): Q of each complete cohort split into within-family, within-rest and between pairs

Source: `q8_families_q_decomposition.csv`. Q is Task 1's definition (mean of the distance over all ordered pairs of the cohort, diagonal included), so the three contributions sum to Q; `change 2020-23` is the block's contribution in 2020-23 minus its mean over the two earliest cohorts (the reference of Task 1's ΔQ).

| family   | cohort   |   designs |   family designs | block         |   share of pairs |   mean distance |   contribution to Q |   contribution, mean of the two earliest cohorts |   change 2020-23 |   Q of cohort |
|:---------|:---------|----------:|-----------------:|:--------------|-----------------:|----------------:|--------------------:|-------------------------------------------------:|-----------------:|--------------:|
| P1       | 2020-23  |       237 |               82 | within family |           0.1197 |          0.3937 |              0.0471 |                                           0.0072 |           0.0399 |        0.5185 |
| P1       | 2020-23  |       237 |               82 | within rest   |           0.4277 |          0.5343 |              0.2286 |                                           0.3895 |          -0.1609 |        0.5185 |
| P1       | 2020-23  |       237 |               82 | between       |           0.4526 |          0.5365 |              0.2428 |                                           0.1497 |           0.0931 |        0.5185 |
| P2       | 2020-23  |       237 |              125 | within family |           0.2782 |          0.4429 |              0.1232 |                                           0.0283 |           0.0949 |        0.5185 |
| P2       | 2020-23  |       237 |              125 | within rest   |           0.2233 |          0.5345 |              0.1194 |                                           0.2903 |          -0.1709 |        0.5185 |
| P2       | 2020-23  |       237 |              125 | between       |           0.4985 |          0.5534 |              0.2759 |                                           0.2278 |           0.0481 |        0.5185 |
| P3       | 2020-23  |       237 |               91 | within family |           0.1474 |          0.4195 |              0.0618 |                                           0.0103 |           0.0515 |        0.5185 |
| P3       | 2020-23  |       237 |               91 | within rest   |           0.3795 |          0.5345 |              0.2028 |                                           0.3699 |          -0.1671 |        0.5185 |
| P3       | 2020-23  |       237 |               91 | between       |           0.4731 |          0.5365 |              0.2538 |                                           0.1662 |           0.0876 |        0.5185 |
| P4       | 2020-23  |       237 |                5 | within family |           0.0004 |          0.4121 |              0.0002 |                                           0      |           0.0002 |        0.5185 |
| P4       | 2020-23  |       237 |                5 | within rest   |           0.9583 |          0.5134 |              0.492  |                                           0.5351 |          -0.0431 |        0.5185 |
| P4       | 2020-23  |       237 |                5 | between       |           0.0413 |          0.6369 |              0.0263 |                                           0.0112 |           0.0151 |        0.5185 |
| P5       | 2020-23  |       237 |              140 | within family |           0.3489 |          0.4534 |              0.1582 |                                           0.0358 |           0.1224 |        0.5185 |
| P5       | 2020-23  |       237 |              140 | within rest   |           0.1675 |          0.5364 |              0.0899 |                                           0.2683 |          -0.1784 |        0.5185 |
| P5       | 2020-23  |       237 |              140 | between       |           0.4835 |          0.5592 |              0.2704 |                                           0.2422 |           0.0282 |        0.5185 |
